# MuJoCo 20 分钟直觉实验

今天只研究一条单关节机械臂。目标不是记住接口，而是亲眼看见：

**模型规定世界怎样运动 → 状态记录现在怎样 → 控制输入一个动作 → 仿真向前走一步。**

从上到下逐格运行即可。暂时不要打开完整项目，也不训练神经网络。

## 1. 准备环境

本地环境已经安装完成。下面的检查也让这份文件可以上传到 Google Colab 后直接运行。

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("mujoco") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "mujoco==3.10.0", "mediapy>=1.2,<2", "ipywidgets>=8,<9"
    ])

import math

import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np

print("环境正常")
print("MuJoCo 版本：", mujoco.__version__)

## 2. 用一小段文字造出机械臂

下面的 XML（可扩展标记语言）就是模型：它写了重力、关节、电机和机械臂的形状。先不用逐行理解。

In [ ]:
ARM_XML = r"""
<mujoco model="one_joint_arm">
  <compiler angle="degree"/>
  <option timestep="0.002" gravity="0 0 -9.81" integrator="implicitfast"/>
  <visual><headlight ambient="0.4 0.4 0.4" diffuse="0.7 0.7 0.7"/></visual>
  <worldbody>
    <camera name="view" pos="1.6 -2.2 1.4" xyaxes="0.81 0.59 0 -0.28 0.39 0.88"/>
    <geom type="plane" size="3 3 0.1" rgba="0.9 0.9 0.92 1"/>
    <body pos="0 0 0.8">
      <geom type="sphere" size="0.075" rgba="0.15 0.15 0.18 1"/>
      <body>
        <joint name="shoulder" type="hinge" axis="0 1 0" damping="0.15"/>
        <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.045"
              density="650" rgba="0.2 0.55 0.9 1"/>
        <site name="tip" pos="0 0 -0.5" size="0.025" rgba="1 0.25 0.1 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <motor name="motor" joint="shoulder" ctrlrange="-5 5" ctrllimited="true"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(ARM_XML)  # 不变的规则书
data = mujoco.MjData(model)                     # 不断变化的现场记录

print("关节位置数量：", model.nq)
print("关节速度数量：", model.nv)
print("电机控制数量：", model.nu)
print("每一步经过：", model.opt.timestep, "秒")

## 3. 先看一眼现在的世界

把机械臂放在 70°，但还不让时间向前走。

In [ ]:
data.qpos[0] = math.radians(70)  # 当前关节角度
mujoco.mj_forward(model, data)   # 根据角度算出机械臂在画面中的位置

with mujoco.Renderer(model, height=480, width=640) as renderer:
    renderer.update_scene(data, camera="view")
    frame = renderer.render().copy()

media.show_image(frame)

## 4. 同一个身体，两种命运

蓝色实验不给电机指令，让机械臂只受重力影响。橙色实验让控制器把它拉到 30°。

In [ ]:
def simulate(initial_deg=70, target_deg=30, damping=0.15, controlled=False, video=False):
    model = mujoco.MjModel.from_xml_string(ARM_XML)
    model.dof_damping[0] = damping
    data = mujoco.MjData(model)
    data.qpos[0] = math.radians(initial_deg)
    mujoco.mj_forward(model, data)

    target = math.radians(target_deg)
    times, angles, controls, frames = [], [], [], []
    renderer = mujoco.Renderer(model, height=360, width=480) if video else None
    frame_stride = max(1, round((1 / 30) / model.opt.timestep))
    step = 0

    while data.time < 3.0:
        if controlled:
            feedback = 12 * (target - data.qpos[0]) - 3 * data.qvel[0]
            gravity_help = data.qfrc_bias[0]
            data.ctrl[0] = np.clip(feedback + gravity_help, -5, 5)
        else:
            data.ctrl[0] = 0

        mujoco.mj_step(model, data)
        times.append(data.time)
        angles.append(math.degrees(data.qpos[0]))
        controls.append(data.ctrl[0])

        if renderer is not None and step % frame_stride == 0:
            renderer.update_scene(data, camera="view")
            frames.append(renderer.render().copy())
        step += 1

    if renderer is not None:
        renderer.close()
    return np.array(times), np.array(angles), np.array(controls), frames

free_time, free_angle, _, _ = simulate(controlled=False)
ctrl_time, ctrl_angle, ctrl_signal, ctrl_frames = simulate(controlled=True, video=True)

print("有控制的机械臂：")
media.show_video(ctrl_frames, fps=30)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(free_time, free_angle, label="没有控制：自由摆动")
plt.plot(ctrl_time, ctrl_angle, label="有控制：去 30°")
plt.axhline(30, color="black", linestyle="--", label="目标")
plt.xlabel("仿真时间（秒）")
plt.ylabel("关节角度（度）")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print(f"没有控制，最后停在 {free_angle[-1]:.1f}° 附近")
print(f"有控制，最后到达 {ctrl_angle[-1]:.1f}°")

## 5. 用滑块玩，不改代码

拖动初始角度、目标角度和阻尼。每次先猜曲线会怎样，再移动滑块。阻尼可以理解为“运动时有多难甩动”。

In [ ]:
from ipywidgets import FloatSlider, interact

@interact(
    initial_deg=FloatSlider(value=70, min=-120, max=120, step=10, description="初始角度"),
    target_deg=FloatSlider(value=30, min=-90, max=90, step=10, description="目标角度"),
    damping=FloatSlider(value=0.15, min=0.0, max=1.0, step=0.05, description="阻尼"),
)
def try_it(initial_deg, target_deg, damping):
    time, angle, _, _ = simulate(
        initial_deg=initial_deg, target_deg=target_deg,
        damping=damping, controlled=True
    )
    plt.figure(figsize=(8, 3))
    plt.plot(time, angle)
    plt.axhline(target_deg, color="black", linestyle="--")
    plt.xlabel("时间（秒）")
    plt.ylabel("角度（度）")
    plt.grid(alpha=0.25)
    plt.show()
    print(f"最终角度：{angle[-1]:.2f}°")

## 6. 今天到这里就够了

如果现在能说清下面四句话，第一课就完成：

1. `model` 是不变的物理规则。
2. `data` 是不断变化的当前状态。
3. `ctrl` 是电机收到的动作。
4. `mj_step` 让世界向前走一步。

下一份再看接触和机器人学习任务接口。想看完整材料时，使用 [MuJoCo 官方 Colab 教程](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb)。